In [ ]:
!pip install transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1"

In [ ]:
from huggingface_hub import login

hf_token = "your_hf_token"

login(token=hf_token)
print("Successfully logged into Hugging Face Hub!")

Successfully logged into Hugging Face Hub!


Loading the model

In [ ]:
from transformers import AutoModel
import torch
import torchaudio

# 1. Load the model
print("Loading the model... (This will take a few minutes on first run)")
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")



Loading the model... (This will take a few minutes on first run)
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:115: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model loaded on cuda


Loading datasets from huggingface

In [ ]:
!pip install -q datasets huggingface_hub

from datasets import load_dataset

bengali_valid = load_dataset(
    "ai4bharat/IndicVoices",
    "bengali",
    split="valid",        # the split name inside the config; usually “train” for the data split
    streaming=True       # streams rows on‑demand
)


Resolving data files:   0%|          | 0/112 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

Calculating no. of reference characters in the dataset

In [ ]:
import pandas as pd

!pip install -q datasets huggingface_hub

from datasets import load_dataset

bengali_valid_streaming = load_dataset(
    "ai4bharat/IndicVoices",
    "bengali",
    split="valid",
    streaming=True
)

# Extract transcriptions and store them in a list
transcriptions = []
for sample in bengali_valid_streaming:
    transcriptions.append(sample["text"])

# Create a pandas DataFrame from the transcriptions
df_transcriptions = pd.DataFrame(transcriptions, columns=['transcription'])

print(f"Successfully loaded {len(df_transcriptions)} transcriptions into a DataFrame.")
display(df_transcriptions.head())

Resolving data files:   0%|          | 0/112 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

Successfully loaded 4009 transcriptions into a DataFrame.


,transcription
0,পূর্ব বর্ধমান পশ্চিম বর্ধমান বাঁকুড়া বীরভূম হ...
1,বন্ধ
2,পাঁচই অগ্রাহণ সাতই জৈষ্ঠ নয়ই মাঘ তেরোই ফাল্গু...
3,এটি প্রাথমিক ও মাধ্যমিক বিদ্যালয়ের শিশুদের জন...
4,পর্যবেক্ষণ করা আমার সব থেকে প্রিয় পাখি হয়েছে...


Calculating no. of reference characters for first set(1975 samples)

In [ ]:
# Calculate the total number of reference characters in the first 1975 samples
num_samples_first_segment = 1975


if len(df_transcriptions) < num_samples_first_segment:
    print(f"Warning: DataFrame has only {len(df_transcriptions)} samples, calculating for all available samples.")
    samples_to_count = df_transcriptions['transcription'].tolist()
else:
    samples_to_count = df_transcriptions['transcription'].head(num_samples_first_segment).tolist()

total_chars_first_1975_samples = sum(len(s) for s in samples_to_count)

print(f"Total number of reference characters in the first {len(samples_to_count)} samples: {total_chars_first_1975_samples}")

Total number of reference characters in the first 1975 samples: 157441


Calculating no. of reference characters for second set(1931 samples)

In [ ]:
# Calculate the total number of reference characters for the second 1931 samples
start_index_second_segment = 1975 # This is the 1976th sample
num_samples_second_segment = 1931

# Get the segment of transcriptions
second_segment_transcriptions = df_transcriptions['transcription'].iloc[start_index_second_segment : start_index_second_segment + num_samples_second_segment].tolist()

total_chars_second_1931_samples = sum(len(s) for s in second_segment_transcriptions)

print(f"Total number of reference characters in the second {len(second_segment_transcriptions)} samples (from sample {start_index_second_segment + 1} to {start_index_second_segment + len(second_segment_transcriptions)}): {total_chars_second_1931_samples}")

Total number of reference characters in the second 1931 samples (from sample 1976 to 3906): 135458


In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.6 MB/s eta 0:00:00


In [ ]:
import numpy as np

Running inference for first set

In [ ]:
import numpy as np # Add this import to ensure np is defined
import torch
import torchaudio

# Definition of load_audio2 function
def load_audio2(audio_source, target_sr=16000):
    if isinstance(audio_source, tuple) and len(audio_source) == 2:
        waveform_np, original_sr = audio_source
        waveform = torch.from_numpy(waveform_np).float().unsqueeze(0)
    else:
        waveform, original_sr = torchaudio.load(audio_source)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    if original_sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=original_sr, new_freq=target_sr)
        waveform = resampler(waveform)
    return waveform

language_code = "bn" # Correct language code for Bengali

# Initialize lists to store references and predictions for the first set samples
sample_references = []
sample_predictions_ctc = []
sample_predictions_rnnt = []

total_samples = 3906
first_set_samples = bengali_valid.take(1975)

print(f"Running inference for the first {1975} samples...")

for sample_idx, sample in enumerate(first_set_samples):
    audio_data_decoded = sample["audio_filepath"]
    reference_text = sample["text"]

    try:
        # Ensure audio_data_decoded['array'] is a numpy array for load_audio2
        audio_waveform_np = np.array(audio_data_decoded['array'])
        audio_input = load_audio2((audio_waveform_np, audio_data_decoded['sampling_rate'])).to(device)

        with torch.no_grad():
            transcription_ctc  = model(audio_input, language_code, "ctc")
            transcription_rnnt = model(audio_input, language_code, "rnnt")

        sample_references.append(reference_text)
        sample_predictions_ctc.append(transcription_ctc.strip())
        sample_predictions_rnnt.append(transcription_rnnt.strip())

        print(f"\n--- Sample {sample_idx + 1} ---")
        print(f"Reference: {reference_text}")
        print(f"CTC Prediction: {transcription_ctc.strip()}")
        print(f"RNN-T Prediction: {transcription_rnnt.strip()}")

    except RuntimeError as e:
        print(f"Skipping sample {sample_idx + 1} due to audio processing error: {e}.")
        # Still append empty strings to keep lists aligned, or handle as needed
        sample_references.append(reference_text)
        sample_predictions_ctc.append("[ERROR]")
        sample_predictions_rnnt.append("[ERROR]")

Streaming output truncated to the last 5000 lines.

--- Sample 976 ---
Reference: আচ্ছা আমাদের এখানে কিন্তু ব্লাউজের বেশ ভ্যারাইটি এসেছে আপনি এসে দেখবেন <unintelligible> আশা করছি পছন্দ হবে কেমন হুম আচ্ছা ঠিক আছে আপনি তাহলে আসুন একদিন আমাকে আসার আগে একবার ফোন করে দেবেন কেমন আচ্ছা আচ্ছা
CTC Prediction: আচ্ছা আমাদের এখানে কিন্তু ব্লাউজের বেশ ভ্যারাইটি এসছে আপনি এসে দেখবেন আশা করছি পছন্দ হবে কেমন হুম আচ্ছা ঠিক আছে আপনি তাহলে আসুন একদিন আমাকে আসার আ একবার ফোন করে দেবেন কেমন আচ্ছা আচ্ছা
RNN-T Prediction: আচ্ছা আমাদের এখানে কিন্তু ব্লাউজের বেশ ভ্যারাইটি এসছে আপনি এসে দেখবেন আশা করছি পছন্দ হবে কেমন হুম আচ্ছা ঠিক আছে আপনি তাহলে আসুন একদিন আমাকে আসার আগে একবার ফোন করে দেবেন কেমন আচ্ছা আচ্ছা

--- Sample 977 ---
Reference: আপনি কি নতুন ব্যবসা শুরু করার সাথে জড়িত অনুষ্ঠানগুলির বর্ণনা করতে পারেন হ্যাঁ আমি অনেকগুলো নতুন ব্যবসা শুরু করা অনুষ্ঠানে জি গিয়েছিলাম
CTC Prediction: আপনি কি নতুন ব্যবসা শুরু করার সাথে জড়িত অনুষ্ঠানগুলির বর্ণনা করতে পারেন হ্যাঁ আমি অনেকগুলো নতুন ব্যবসা শুরু করা অনুষ্ঠানের দি

KeyboardInterrupt: 

Calculating S,D,I,N for first set (Word-level)

In [ ]:
from jiwer import process_words

# For CTC
total_substitutions_ctc = 0
total_deletions_ctc = 0
total_insertions_ctc = 0
total_reference_words_ctc = 0

for ref, pred in zip(sample_references, sample_predictions_ctc):
    metrics = process_words([ref], [pred])
    total_substitutions_ctc += metrics.substitutions
    total_deletions_ctc += metrics.deletions
    total_insertions_ctc += metrics.insertions
    # Calculate reference words for individual sentences
    total_reference_words_ctc += len(ref.split())

# For RNNT
total_substitutions_rnnt = 0
total_deletions_rnnt = 0
total_insertions_rnnt = 0
total_reference_words_rnnt = 0

for ref, pred in zip(sample_references, sample_predictions_rnnt):
    metrics = process_words([ref], [pred])
    total_substitutions_rnnt += metrics.substitutions
    total_deletions_rnnt += metrics.deletions
    total_insertions_rnnt += metrics.insertions
    # Calculate reference words for individual sentences
    total_reference_words_rnnt += len(ref.split())


print("\n--- CTC Model Results ---")
print(f"Total reference words: {total_reference_words_ctc}")
print(f"Total substitutions: {total_substitutions_ctc}")
print(f"Total deletions: {total_deletions_ctc}")
print(f"Total insertions: {total_insertions_ctc}")

print("\n--- RNNT Model Results ---")
print(f"Total reference words: {total_reference_words_rnnt}")
print(f"Total substitutions: {total_substitutions_rnnt}")
print(f"Total deletions: {total_deletions_rnnt}")
print(f"Total insertions: {total_insertions_rnnt}")


--- CTC Model Results ---
Total reference words: 28240
Total substitutions: 2802
Total deletions: 436
Total insertions: 409

--- RNNT Model Results ---
Total reference words: 28240
Total substitutions: 2574
Total deletions: 462
Total insertions: 414


Calculating S,D,I,N for first set (Char-level)

In [ ]:
from jiwer import process_characters

# For CTC
total_substitutions_ctc = 0
total_deletions_ctc = 0
total_insertions_ctc = 0
total_reference_words_ctc = 0

for ref, pred in zip(sample_references, sample_predictions_ctc):
    metrics = process_characters([ref], [pred])
    total_substitutions_ctc += metrics.substitutions
    total_deletions_ctc += metrics.deletions
    total_insertions_ctc += metrics.insertions
    # Calculate reference words for individual sentences
    total_reference_words_ctc += len(ref.split())

# For RNNT
total_substitutions_rnnt = 0
total_deletions_rnnt = 0
total_insertions_rnnt = 0
total_reference_words_rnnt = 0

for ref, pred in zip(sample_references, sample_predictions_rnnt):
    metrics = process_characters([ref], [pred])
    total_substitutions_rnnt += metrics.substitutions
    total_deletions_rnnt += metrics.deletions
    total_insertions_rnnt += metrics.insertions
    # Calculate reference words for individual sentences
    total_reference_words_rnnt += len(ref.split())


print("\n--- CTC Model Results ---")
print(f"Total reference words: {total_reference_words_ctc}")
print(f"Total substitutions: {total_substitutions_ctc}")
print(f"Total deletions: {total_deletions_ctc}")
print(f"Total insertions: {total_insertions_ctc}")

print("\n--- RNNT Model Results ---")
print(f"Total reference words: {total_reference_words_rnnt}")
print(f"Total substitutions: {total_substitutions_rnnt}")
print(f"Total deletions: {total_deletions_rnnt}")
print(f"Total insertions: {total_insertions_rnnt}")


--- CTC Model Results ---
Total reference words: 28240
Total substitutions: 2045
Total deletions: 3633
Total insertions: 2007

--- RNNT Model Results ---
Total reference words: 28240
Total substitutions: 2204
Total deletions: 3517
Total insertions: 1870


Running inference for second set

In [ ]:
import numpy as np # Add this import to ensure np is defined
import torch
import torchaudio

# Definition of load_audio2 function
def load_audio2(audio_source, target_sr=16000):
    if isinstance(audio_source, tuple) and len(audio_source) == 2:
        waveform_np, original_sr = audio_source
        waveform = torch.from_numpy(waveform_np).float().unsqueeze(0)
    else:
        waveform, original_sr = torchaudio.load(audio_source)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    if original_sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=original_sr, new_freq=target_sr)
        waveform = resampler(waveform)
    return waveform

language_code = "bn" # Correct language code for Bengali

# Initialize lists to store references and predictions for the second set samples
sample_references = []
sample_predictions_ctc = []
sample_predictions_rnnt = []


total_samples = 3906
second_set_samples = bengali_valid.skip(1975).take(1931)

print(f"Running inference for the second {1931} samples...")

for sample_idx, sample in enumerate(second_set_samples):
    audio_data_decoded = sample["audio_filepath"]
    reference_text = sample["text"]

    try:
        # Ensure audio_data_decoded['array'] is a numpy array for load_audio2
        audio_waveform_np = np.array(audio_data_decoded['array'])
        audio_input = load_audio2((audio_waveform_np, audio_data_decoded['sampling_rate'])).to(device)

        with torch.no_grad():
            transcription_ctc  = model(audio_input, language_code, "ctc")
            transcription_rnnt = model(audio_input, language_code, "rnnt")

        sample_references.append(reference_text)
        sample_predictions_ctc.append(transcription_ctc.strip())
        sample_predictions_rnnt.append(transcription_rnnt.strip())

        print(f"\n--- Sample {sample_idx + 1} ---")
        print(f"Reference: {reference_text}")
        print(f"CTC Prediction: {transcription_ctc.strip()}")
        print(f"RNN-T Prediction: {transcription_rnnt.strip()}")

    except RuntimeError as e:
        print(f"Skipping sample {sample_idx + 1} due to audio processing error: {e}.")
        # Still append empty strings to keep lists aligned, or handle as needed
        sample_references.append(reference_text)
        sample_predictions_ctc.append("[ERROR]")
        sample_predictions_rnnt.append("[ERROR]")

Streaming output truncated to the last 5000 lines.

--- Sample 932 ---
Reference: মাসিমনি ফোন করেছিল নিমন্ত্রণ করেছে
CTC Prediction: মাসেমনি ফোন করেছিল নিমন্ত্রণা করেছে
RNN-T Prediction: মাসে উনি ফোন করেছিল নিমন্ত্রণা করেছে

--- Sample 933 ---
Reference: আমার ছেলে মেয়ে কী করছে ছেলে খেলতে গেছে আর মেয়েও গান নিতে গিয়েছে গানের টিউশনে
CTC Prediction: আমার ছেলে মেয়ে কী করছে ছে ছেলে খেলতে গেছে আর মেয়েও গান নিতে আসছে গানে টিউশ নিতে
RNN-T Prediction: আমার ছেলে মেয়ে কী করছে ছেলে খেলতে গেছে আর মেয়েও গান নিতে আসে গানে টিউশ নিতে

--- Sample 934 ---
Reference: আসো
CTC Prediction: আচ্ছা
RNN-T Prediction: আচ্ছা

--- Sample 935 ---
Reference: ঠিক আছে
CTC Prediction: ঠিক আছে
RNN-T Prediction: ঠিক আছে

--- Sample 936 ---
Reference: ব্যাংকক হংকং দুবাই মেক্সিকো প্যারিস
CTC Prediction: ব্যাংকক হংকং দুবাই মেক্সিকো প্যারিস
RNN-T Prediction: ব্যাংকক হংকং দুবাই মেক্সিকো প্যারিস

--- Sample 937 ---
Reference: তালিকাগুলো তালিকা করো
CTC Prediction: তালিকাগুলো তালিকা করো
RNN-T Prediction: তালিকাগুলো তালিকা ক

In [ ]:
import pandas as pd

# Create a DataFrame from the sample_references list
df_references = pd.DataFrame(sample_references, columns=['reference_text'])

# Display the first 5 rows of the DataFrame
display(df_references.head())

,reference_text
0,বদল খাবারগুলো দিন নয় টাকাটা রিটার্ন দিন
1,যে কোনো একটা তো করুন
2,বিভিন্ন গোষ্ঠীর বা জাতির মাধ্যমে যে একটা খেলা ...
3,মাঠে নেমে অনেকের সাথে হাত মেলনো হয়
4,সেটা ভালো লাগে সবার সাথে তারপর খেলা হয় খেলা হ...


In [ ]:
len(df_references)

1931

Calculating S,D,I,N for second set(word-level)

In [ ]:
from jiwer import process_words

# For CTC
total_substitutions_ctc = 0
total_deletions_ctc = 0
total_insertions_ctc = 0
total_reference_words_ctc = 0

for ref, pred in zip(sample_references, sample_predictions_ctc):
    metrics = process_words([ref], [pred])
    total_substitutions_ctc += metrics.substitutions
    total_deletions_ctc += metrics.deletions
    total_insertions_ctc += metrics.insertions
    # Calculate reference words for individual sentences
    total_reference_words_ctc += len(ref.split())

# For RNNT
total_substitutions_rnnt = 0
total_deletions_rnnt = 0
total_insertions_rnnt = 0
total_reference_words_rnnt = 0

for ref, pred in zip(sample_references, sample_predictions_rnnt):
    metrics = process_words([ref], [pred])
    total_substitutions_rnnt += metrics.substitutions
    total_deletions_rnnt += metrics.deletions
    total_insertions_rnnt += metrics.insertions
    # Calculate reference words for individual sentences
    total_reference_words_rnnt += len(ref.split())


print("\n--- CTC Model Results ---")
print(f"Total reference words: {total_reference_words_ctc}")
print(f"Total substitutions: {total_substitutions_ctc}")
print(f"Total deletions: {total_deletions_ctc}")
print(f"Total insertions: {total_insertions_ctc}")

print("\n--- RNNT Model Results ---")
print(f"Total reference words: {total_reference_words_rnnt}")
print(f"Total substitutions: {total_substitutions_rnnt}")
print(f"Total deletions: {total_deletions_rnnt}")
print(f"Total insertions: {total_insertions_rnnt}")


--- CTC Model Results ---
Total reference words: 24591
Total substitutions: 2901
Total deletions: 617
Total insertions: 355

--- RNNT Model Results ---
Total reference words: 24591
Total substitutions: 2684
Total deletions: 664
Total insertions: 330


Calculating S,D,I,N for second set(char-level)

In [ ]:
from jiwer import process_characters

# For CTC
total_substitutions_ctc = 0
total_deletions_ctc = 0
total_insertions_ctc = 0
total_reference_chars_ctc = 0

for ref, pred in zip(sample_references, sample_predictions_ctc):
    metrics = process_characters([ref], [pred])
    total_substitutions_ctc += metrics.substitutions
    total_deletions_ctc += metrics.deletions
    total_insertions_ctc += metrics.insertions
    total_reference_chars_ctc += len(ref.replace(" ", ""))

# For RNNT
total_substitutions_rnnt = 0
total_deletions_rnnt = 0
total_insertions_rnnt = 0
total_reference_chars_rnnt = 0

for ref, pred in zip(sample_references, sample_predictions_rnnt):
    metrics = process_characters([ref], [pred])
    total_substitutions_rnnt += metrics.substitutions
    total_deletions_rnnt += metrics.deletions
    total_insertions_rnnt += metrics.insertions
    total_reference_chars_ctc += len(ref.replace(" ", ""))


print("\n--- CTC Model Results ---")
print(f"Total reference words: {total_reference_chars_ctc}")
print(f"Total substitutions: {total_substitutions_ctc}")
print(f"Total deletions: {total_deletions_ctc}")
print(f"Total insertions: {total_insertions_ctc}")

print("\n--- RNNT Model Results ---")
print(f"Total reference words: {total_reference_chars_rnnt}")
print(f"Total substitutions: {total_substitutions_rnnt}")
print(f"Total deletions: {total_deletions_rnnt}")
print(f"Total insertions: {total_insertions_rnnt}")


--- CTC Model Results ---
Total reference words: 24591
Total substitutions: 2558
Total deletions: 3833
Total insertions: 1780

--- RNNT Model Results ---
Total reference words: 24591
Total substitutions: 2630
Total deletions: 3855
Total insertions: 1626
